## Construcción de la capa analítica

### Objetivo

El objetivo de este notebook es transformar los datos transaccionales originales en una capa analítica limpia y reproducible que pueda utilizarse posteriormente en Python, SQL, Power BI y Strategy/MicroStrategy.

Las reglas aplicadas se basan en los hallazgos obtenidos durante la auditoría de datos.

En esta fase se construirán métricas y campos derivados relacionados con:

- unidades pedidas;
- unidades devueltas;
- unidades retenidas;
- precio de venta;
- RRP;
- markdown;
- estado comercial del producto;
- uso de vouchers;
- validez de las transacciones.

Los datos originales no serán modificados.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
DATA_PATH = Path("../data/raw")

train_path = DATA_PATH / "orders_train.txt"

train = pd.read_csv(
    train_path,
    sep=";",
    na_values="NA"
)

train.shape

(2325165, 15)

In [4]:
df = train.copy()

In [5]:
df["orderDate"] = pd.to_datetime(
    df["orderDate"],
    errors="coerce"
)

In [6]:
df["orderDate"].agg(["min", "max"])

min   2014-01-01
max   2015-09-30
Name: orderDate, dtype: datetime64[us]

In [7]:
df["orderDate"].isna().sum()

np.int64(0)

In [8]:
df["is_zero_quantity"] = df["quantity"] == 0

df["is_invalid_return"] = (
    df["returnQuantity"] > df["quantity"]
)

df["is_zero_value_item"] = (
    (df["quantity"] > 0) &
    (df["price"] == 0) &
    (df["rrp"] == 0)
)

df["has_missing_product_metadata"] = (
    df["rrp"].isna() |
    df["productGroup"].isna()
)

In [9]:
df[
    [
        "is_zero_quantity",
        "is_invalid_return",
        "is_zero_value_item",
        "has_missing_product_metadata"
    ]
].sum()

is_zero_quantity                31643
is_invalid_return                   9
is_zero_value_item              15200
has_missing_product_metadata      351
dtype: int64

In [10]:
df["is_valid_transaction"] = (
    (df["quantity"] > 0) &
    (df["returnQuantity"] <= df["quantity"])
)

In [11]:
df["is_valid_transaction"].value_counts()

is_valid_transaction
True     2293521
False      31644
Name: count, dtype: int64

In [12]:
df["ordered_units"] = df["quantity"]

df["returned_units"] = df["returnQuantity"]

df["kept_units"] = (
    df["ordered_units"] - df["returned_units"]
)

In [13]:
df.loc[
    df["is_valid_transaction"],
    ["ordered_units", "returned_units", "kept_units"]
].describe()

,ordered_units,returned_units,kept_units
count,2.293521e+06,2.293521e+06,2.293521e+06
mean,1.009278e+00,5.285973e-01,4.806810e-01
std,1.158175e-01,5.033689e-01,5.125786e-01
min,1.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,0.000000e+00,0.000000e+00
50%,1.000000e+00,1.000000e+00,0.000000e+00
75%,1.000000e+00,1.000000e+00,1.000000e+00
max,2.400000e+01,5.000000e+00,2.400000e+01


In [14]:
(df.loc[
    df["is_valid_transaction"],
    "kept_units"
] < 0).sum()

np.int64(0)

In [15]:
df[
    [
        "is_zero_quantity",
        "is_invalid_return",
        "is_zero_value_item",
        "has_missing_product_metadata"
    ]
].sum()

is_zero_quantity                31643
is_invalid_return                   9
is_zero_value_item              15200
has_missing_product_metadata      351
dtype: int64

In [16]:
df["is_valid_transaction"].value_counts()

is_valid_transaction
True     2293521
False      31644
Name: count, dtype: int64

In [18]:
df["unit_selling_price"] = np.where(
    df["quantity"] > 0,
    df["price"] / df["quantity"],
    np.nan
)

In [19]:
df.loc[
    df["is_valid_transaction"],
    ["quantity", "price", "unit_selling_price", "rrp"]
].head(20)

,quantity,price,unit_selling_price,rrp
0,1,10.00,10.00,29.99
1,1,20.00,20.00,39.99
2,1,35.00,35.00,49.99
3,1,49.99,49.99,49.99
4,1,10.00,10.00,35.99
5,1,10.00,10.00,35.99
6,1,25.00,25.00,39.99
7,1,15.00,15.00,39.99
9,1,89.99,89.99,89.99
10,1,15.00,15.00,39.99


In [20]:
df["rrp_line_value"] = (
    df["rrp"] * df["quantity"]
)

In [22]:
conditions = [
    ~df["is_valid_transaction"],

    df["is_zero_value_item"],

    df["rrp"].isna(),

    np.isclose(
        df["price"],
        df["rrp_line_value"],
        equal_nan=False
    ),

    df["price"] < df["rrp_line_value"],

    df["price"] > df["rrp_line_value"]
]

choices = [
    "invalid_transaction",
    "zero_value",
    "unknown_rrp",
    "full_price",
    "below_rrp",
    "above_rrp"
]

df["price_status"] = np.select(
    conditions,
    choices,
    default="unknown"
)

In [23]:
df["price_status"].value_counts()

price_status
full_price             1497697
below_rrp               774503
invalid_transaction      31644
zero_value               15200
above_rrp                 5770
unknown_rrp                351
Name: count, dtype: int64

In [24]:
df["price_status"].value_counts(normalize=True)

price_status
full_price             0.644125
below_rrp              0.333096
invalid_transaction    0.013609
zero_value             0.006537
above_rrp              0.002482
unknown_rrp            0.000151
Name: proportion, dtype: float64

In [25]:
df["markdown_amount"] = np.where(
    df["price_status"] == "below_rrp",
    df["rrp_line_value"] - df["price"],
    0.0
)

df.loc[
    df["price_status"] == "unknown_rrp",
    "markdown_amount"
] = np.nan

In [26]:
df["markdown_pct"] = np.where(
    (df["price_status"] == "below_rrp") &
    (df["rrp_line_value"] > 0),
    df["markdown_amount"] / df["rrp_line_value"],
    0.0
)

df.loc[
    df["price_status"] == "unknown_rrp",
    "markdown_pct"
] = np.nan

In [27]:
df.loc[
    df["price_status"] == "below_rrp",
    [
        "price",
        "rrp_line_value",
        "markdown_amount",
        "markdown_pct"
    ]
].describe()

,price,rrp_line_value,markdown_amount,markdown_pct
count,774503.000000,774503.000000,774503.000000,774503.000000
mean,24.986094,46.809209,21.823115,0.460329
std,16.187037,26.090876,15.230226,0.162993
min,3.000000,9.990000,1.600000,0.100010
25%,15.000000,29.990000,9.990000,0.307426
50%,20.000000,39.990000,19.990000,0.499833
75%,30.000000,59.990000,27.990000,0.500139
max,600.000000,799.990000,559.990000,0.874969


In [28]:
df["has_voucher"] = df["voucherAmount"] > 0

df["has_voucher"].value_counts()

has_voucher
False    1917678
True      407487
Name: count, dtype: int64

In [29]:
df["ordered_sales_value"] = np.where(
    df["is_valid_transaction"],
    df["price"],
    np.nan
)

In [30]:
df["returned_sales_value"] = np.where(
    df["is_valid_transaction"],
    df["unit_selling_price"] * df["returned_units"],
    np.nan
)

In [31]:
df["kept_sales_value"] = np.where(
    df["is_valid_transaction"],
    df["unit_selling_price"] * df["kept_units"],
    np.nan
)

In [34]:
value_check = df.loc[
    df["is_valid_transaction"],
    [
        "ordered_sales_value",
        "returned_sales_value",
        "kept_sales_value"
    ]
].copy()

value_check["difference"] = (
    value_check["ordered_sales_value"]
    - value_check["returned_sales_value"]
    - value_check["kept_sales_value"]
)

value_check["difference"].describe()

count    2.293521e+06
mean    -2.648827e-19
std      1.505216e-16
min     -1.421085e-14
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.842171e-14
Name: difference, dtype: float64

In [38]:
np.isclose(
    value_check["difference"],
    0,
    atol=1e-8
).sum()

np.int64(2293521)

In [39]:
df.loc[
    df["is_valid_transaction"],
    [
        "ordered_sales_value",
        "returned_sales_value",
        "kept_sales_value"
    ]
].sum()

ordered_sales_value     82308730.38
returned_sales_value    46786190.79
kept_sales_value        35522539.59
dtype: float64

In [40]:
(df.loc[
    df["is_valid_transaction"],
    "returned_sales_value"
] > df.loc[
    df["is_valid_transaction"],
    "ordered_sales_value"
]).sum()

np.int64(4)

In [41]:
value_anomalies = df.loc[
    df["is_valid_transaction"]
    & (df["returned_sales_value"] > df["ordered_sales_value"]),
    [
        "orderID",
        "articleID",
        "quantity",
        "returnQuantity",
        "price",
        "unit_selling_price",
        "ordered_sales_value",
        "returned_sales_value",
        "kept_sales_value"
    ]
].copy()

value_anomalies["difference"] = (
    value_anomalies["returned_sales_value"]
    - value_anomalies["ordered_sales_value"]
)

value_anomalies

,orderID,articleID,quantity,returnQuantity,price,unit_selling_price,ordered_sales_value,returned_sales_value,kept_sales_value,difference
944593,a1299532,i1000437,5,5,99.95,19.99,99.95,99.95,0.0,1.421085e-14
1180581,a1378132,i1000397,5,5,89.95,17.99,89.95,89.95,0.0,1.421085e-14
1180582,a1378132,i1000397,5,5,89.95,17.99,89.95,89.95,0.0,1.421085e-14
1224151,a1393512,i1000397,5,5,89.95,17.99,89.95,89.95,0.0,1.421085e-14


In [42]:
value_anomalies["difference"].describe()

count    4.000000e+00
mean     1.421085e-14
std      0.000000e+00
min      1.421085e-14
25%      1.421085e-14
50%      1.421085e-14
75%      1.421085e-14
max      1.421085e-14
Name: difference, dtype: float64

In [43]:
real_value_anomalies = (
    (df["returned_sales_value"] > df["ordered_sales_value"])
    &
    ~np.isclose(
        df["returned_sales_value"],
        df["ordered_sales_value"],
        atol=1e-8,
        equal_nan=False
    )
)

real_value_anomalies.sum()

np.int64(0)

In [44]:
valid_df = df.loc[
    df["is_valid_transaction"]
].copy()

valid_df.shape

(2293521, 32)

In [45]:
unit_totals = valid_df[
    [
        "ordered_units",
        "returned_units",
        "kept_units"
    ]
].sum()

unit_totals

ordered_units     2314801
returned_units    1212349
kept_units        1102452
dtype: int64

In [46]:
unit_return_rate = (
    valid_df["returned_units"].sum()
    / valid_df["ordered_units"].sum()
)

unit_retention_rate = (
    valid_df["kept_units"].sum()
    / valid_df["ordered_units"].sum()
)

unit_return_rate, unit_retention_rate

(np.float64(0.5237378936677494), np.float64(0.4762621063322506))

In [47]:
unit_return_rate + unit_retention_rate

np.float64(1.0)

In [48]:
value_return_rate = (
    valid_df["returned_sales_value"].sum()
    / valid_df["ordered_sales_value"].sum()
)

value_retention_rate = (
    valid_df["kept_sales_value"].sum()
    / valid_df["ordered_sales_value"].sum()
)

value_return_rate, value_retention_rate

(np.float64(0.5684231863861731), np.float64(0.431576813613827))

In [49]:
value_return_rate + value_retention_rate

np.float64(1.0)

In [50]:
global_kpis = pd.Series({
    "ordered_units": valid_df["ordered_units"].sum(),
    "returned_units": valid_df["returned_units"].sum(),
    "kept_units": valid_df["kept_units"].sum(),

    "ordered_sales_value": valid_df["ordered_sales_value"].sum(),
    "returned_sales_value": valid_df["returned_sales_value"].sum(),
    "kept_sales_value": valid_df["kept_sales_value"].sum(),

    "unit_return_rate": unit_return_rate,
    "unit_retention_rate": unit_retention_rate,

    "value_return_rate": value_return_rate,
    "value_retention_rate": value_retention_rate
})

global_kpis

ordered_units           2.314801e+06
returned_units          1.212349e+06
kept_units              1.102452e+06
ordered_sales_value     8.230873e+07
returned_sales_value    4.678619e+07
kept_sales_value        3.552254e+07
unit_return_rate        5.237379e-01
unit_retention_rate     4.762621e-01
value_return_rate       5.684232e-01
value_retention_rate    4.315768e-01
dtype: float64

In [51]:
PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

In [52]:
df.to_parquet(
    PROCESSED_PATH / "analytical_transactions.parquet",
    index=False
)

## Conclusiones de la capa analítica

La capa analítica ha quedado construida a partir de las reglas definidas durante la auditoría de datos, manteniendo separados los datos originales y las variables derivadas.

### 1. Transacciones válidas

Se han identificado **2.293.521 transacciones válidas** para el análisis.

Las líneas con `quantity = 0` y los registros con devoluciones inconsistentes quedan fuera de los KPIs transaccionales, pero se conservan en los datos originales.

### 2. Métricas de unidades

Se han definido tres métricas base:

- `ordered_units`
- `returned_units`
- `kept_units`

Estas métricas permiten separar demanda bruta, devoluciones y demanda finalmente retenida.

### 3. Métricas de valor

Se han creado:

- `ordered_sales_value`
- `returned_sales_value`
- `kept_sales_value`

Estas variables permiten analizar el valor asociado a las unidades compradas, devueltas y retenidas.

`kept_sales_value` no se interpreta como margen ni como revenue neto, ya que el dataset no contiene costes ni suficiente información contable.

### 4. Pricing y markdown

Se han separado los conceptos de:

- precio de venta;
- RRP;
- markdown;
- voucher.

También se han creado categorías de estado de precio como `full_price`, `below_rrp`, `above_rrp`, `zero_value` y `unknown_rrp`.

Esto evita mezclar descuentos de producto con vouchers aplicados a nivel de pedido.

### 5. KPIs globales

Sobre las transacciones válidas:

- **52,34%** de las unidades fueron devueltas.
- **47,66%** de las unidades fueron retenidas.
- **56,84%** del valor de venta asociado a las compras corresponde a unidades devueltas.
- **43,16%** corresponde a unidades retenidas.

La tasa de devolución por valor es superior a la tasa de devolución por unidades, lo que sugiere que las devoluciones están concentrando proporcionalmente más valor económico.

### Resultado

La capa analítica queda preparada para analizar el rendimiento real de producto desde una perspectiva de merchandising.

El siguiente paso será estudiar si los productos que destacan por demanda bruta mantienen su posición cuando se analiza la demanda realmente retenida después de las devoluciones.